# Candidate historical features
Decision-neutral lagged driver, constructor, and circuit candidates.

## Simple explanation

This notebook creates information that was known **before each race** from a driver's, constructor's, or circuit's earlier races. Each row represents one driver in one race. For example, `driver_avg_finish_5` is the driver's average finishing position in their previous five starts; `driver_podium_rate_5` is the fraction of those starts that ended on the podium; and `driver_points_avg_5` is their average points over those starts. The `constructor_*` and `circuit_*` columns use the same idea for the team and track. A `prior_*` column uses all earlier starts, while a `_3`, `_5`, or `_10` column uses only the previous 3, 5, or 10 starts. The `shift(1)` operation is essential: the current race is never included in its own history.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
root=Path.cwd().resolve()
while root!=root.parent and not (root/'data'/'raw').exists(): root=root.parent
raw=root/'data'/'raw'; read=lambda n:pd.read_csv(raw/f'{n}.csv',dtype=str,keep_default_na=False); races,results=read('races'),read('results')
races['year']=pd.to_numeric(races.year); races['date']=pd.to_datetime(races.date,errors='coerce'); results=results.merge(races[['raceId','year','round','date','circuitId']],on='raceId',validate='many_to_one')
for c in ['grid','positionOrder','points','laps']: results[c]=pd.to_numeric(results[c],errors='coerce')
results['podium']=results.positionOrder.le(3); results['started']=(results.grid>0)|((results.grid==0)&(results.laps>0)); results=results[results.started].copy(); results['_grid_sort']=results.grid.where(results.grid>0,np.inf); results['_result_id_n']=pd.to_numeric(results.resultId,errors='coerce'); results=results.sort_values(['date','raceId','driverId','_grid_sort','_result_id_n','resultId']).drop_duplicates(['raceId','driverId'],keep='first').sort_values(['date','raceId','driverId','resultId']).reset_index(drop=True); results['finish_num']=results.positionOrder.where(results.positionOrder>0)

In [2]:
g=results.groupby('driverId',sort=False); results['driver_prior_starts']=g.podium.transform(lambda s:s.shift(1).expanding().count()); results['driver_prior_podiums']=g.podium.transform(lambda s:s.shift(1).cumsum()).fillna(0); results['driver_prior_podium_rate']=results.driver_prior_podiums/results.driver_prior_starts.replace(0,np.nan)
for w in [3,5,10]:
    results[f'driver_avg_finish_{w}']=g.finish_num.transform(lambda s,w=w:s.shift(1).rolling(w,min_periods=1).mean()); results[f'driver_podium_rate_{w}']=g.podium.transform(lambda s,w=w:s.shift(1).rolling(w,min_periods=1).mean()); results[f'driver_points_avg_{w}']=g.points.transform(lambda s,w=w:s.shift(1).rolling(w,min_periods=1).mean()); results[f'driver_dnf_rate_{w}']=g.finish_num.transform(lambda s,w=w:s.shift(1).rolling(w,min_periods=1).apply(lambda z:z.isna().mean()))

In [3]:
for entity,key in [('constructor','constructorId'),('circuit','circuitId')]:
    g=results.groupby(key,sort=False); results[f'{entity}_prior_starts']=g.podium.transform(lambda s:s.shift(1).expanding().count()); results[f'{entity}_prior_podium_rate']=g.podium.transform(lambda s:s.shift(1).expanding().mean()); results[f'{entity}_prior_points_avg']=g.points.transform(lambda s:s.shift(1).expanding().mean())
feature_cols=[c for c in results if c.startswith(('driver_','constructor_','circuit_'))]; candidate=results[['raceId','driverId','constructorId','circuitId','year','podium']+feature_cols].copy(); print(len(feature_cols)); display(candidate[feature_cols].describe().T)
assert not candidate.duplicated(['raceId','driverId']).any(), 'The R2 deduplication should leave one row per race and driver.'
history_check=candidate.driver_prior_podiums.le(candidate.driver_prior_starts)
assert history_check.all(), candidate.loc[~history_check, ['raceId','driverId','driver_prior_starts','driver_prior_podiums']].head()


21


,count,mean,std,min,25%,50%,75%,max
driver_prior_starts,25105.0,67.197132,69.048654,0.0,14.000000,44.000000,99.000000,403.000000
driver_avg_finish_3,24313.0,11.747330,4.956319,1.0,8.333333,12.000000,15.333333,33.000000
driver_podium_rate_3,24313.0,0.137505,0.257718,0.0,0.000000,0.000000,0.333333,1.000000
driver_points_avg_3,24313.0,2.164797,3.837004,0.0,0.000000,0.500000,2.666667,31.000000
driver_dnf_rate_3,24313.0,0.020257,0.092185,0.0,0.000000,0.000000,0.000000,0.500000
driver_avg_finish_5,24313.0,11.750762,4.492360,1.0,8.600000,12.000000,15.000000,33.000000
driver_podium_rate_5,24313.0,0.137204,0.235581,0.0,0.000000,0.000000,0.200000,1.000000
driver_points_avg_5,24313.0,2.157902,3.676497,0.0,0.000000,0.600000,2.600000,28.600000
driver_dnf_rate_5,24313.0,0.028861,0.100206,0.0,0.000000,0.000000,0.000000,0.500000
driver_avg_finish_10,24313.0,11.769173,4.097619,1.0,9.000000,12.000000,14.600000,33.000000
